In [ ]:
import numpy as np
from scipy.stats import norm
import time

In [ ]:
def TrinomialProbs(r, sigma, deltaT, lambda_val=np.sqrt(3)):
    """
    Calculate trinomial tree probabilities using optimal lambda for kurtosis matching
    Solves: 1) Mean matching, 2) Variance matching, 3) Probabilities sum to 1
    """
    nu = r - 0.5 * sigma**2
    deltaX = lambda_val * sigma * np.sqrt(deltaT)

    # System of equations solution
    V = sigma**2 * deltaT + nu**2 * deltaT**2

    p_u = 0.5 * (V / deltaX**2 + nu * deltaT / deltaX)
    p_d = 0.5 * (V / deltaX**2 - nu * deltaT / deltaX)
    p_m = 1 - p_u - p_d

    return p_u, p_m, p_d, deltaX, lambda_val

In [ ]:
def TrinomialEuCall(S0, K, r, sigma, T, N, lambda_val=np.sqrt(3)):
    start_time = time.time()

    deltaT = T/N
    p_u, p_m, p_d, deltaX, lambda_used = TrinomialProbs(r, sigma, deltaT, lambda_val)
    discount = np.exp(-r * deltaT)

    # Set up S values (at maturity)
    Svals = np.zeros(2*N+1)
    Svals[0] = S0 * np.exp(N * deltaX)
    exp_dX = np.exp(-deltaX)

    for j in range(1, 2*N+1):
        Svals[j] = exp_dX * Svals[j-1]

    # Set up lattice and terminal values
    Cvals = np.zeros((2*N+1, N+1))

    for k in range(0, 2*N+1):
        Cvals[k, N] = max(Svals[k] - K, 0)

    for j in range(N-1, -1, -1):
        for i in range(N-j, N+j+1):
            Cvals[i, j] = discount * (p_u * Cvals[i-1, j+1] + p_m * Cvals[i, j+1] + p_d * Cvals[i+1, j+1])

    price = Cvals[N, 0]
    computation_time = time.time() - start_time

    return price, computation_time

def TrinomialEuPut(S0, K, r, sigma, T, N, lambda_val=np.sqrt(3)):
  start_time = time.time()

  deltaT = T/N
  p_u, p_m, p_d, deltaX, lambda_used = TrinomialProbs(r, sigma, deltaT, lambda_val)
  discount = np.exp(-r * deltaT)

  # Set up S values (at maturity)
  Svals = np.zeros(2*N+1)
  Svals[0] = S0 * np.exp(N * deltaX)
  exp_dX = np.exp(-deltaX)

  for j in range(1, 2*N+1):
      Svals[j] = exp_dX * Svals[j-1]

  # Set up lattice and terminal values
  Pvals = np.zeros((2*N+1, N+1))

  for m in range(0, 2*N+1):
      Pvals[m, N] = max(K - Svals[m], 0)

  for j in range(N-1, -1, -1):
      for i in range(N-j, N+j+1):
          Pvals[i, j] = discount * (p_u * Pvals[i-1, j+1] + p_m * Pvals[i, j+1] + p_d * Pvals[i+1, j+1])

  price = Pvals[N, 0]
  computation_time = time.time() - start_time

  return price, computation_time

In [ ]:
def TrinomialAmPut(S0, K, r, sigma, T, N, lambda_val=np.sqrt(3)):
    start_time = time.time()

    deltaT = T/N
    p_u, p_m, p_d, deltaX, lambda_used = TrinomialProbs(r, sigma, deltaT, lambda_val)
    discount = np.exp(-r * deltaT)

    # Set up S values (at maturity)
    Svals = np.zeros(2*N+1)
    Svals[0] = S0 * np.exp(N * deltaX)
    exp_dX = np.exp(-deltaX)

    for j in range(1, 2*N+1):
        Svals[j] = exp_dX * Svals[j-1]

    # Set up lattice and terminal values
    Pvals = np.zeros((2*N+1, N+1))

    for m in range(0, 2*N+1):
        Pvals[m, N] = max(K - Svals[m], 0)

    for j in range(N-1, -1, -1):
        for i in range(N-j, N+j+1):
            hold = discount * (p_u * Pvals[i-1, j+1] + p_m * Pvals[i, j+1] + p_d * Pvals[i+1, j+1])
            Pvals[i, j] = max(K - Svals[i], hold)

    price = Pvals[N, 0]
    computation_time = time.time() - start_time

    return price, computation_time

def TrinomialAmCall(S0, K, r, sigma, T, N, lambda_val=np.sqrt(3)):
    start_time = time.time()

    deltaT = T / N
    p_u, p_m, p_d, deltaX, lambda_used = TrinomialProbs(r, sigma, deltaT, lambda_val)
    discount = np.exp(-r * deltaT)

    # Set up S values (at maturity)
    Svals = np.zeros(2 * N + 1)
    Svals[0] = S0 * np.exp(N * deltaX)
    exp_dX = np.exp(-deltaX)

    for j in range(1, 2 * N + 1):
        Svals[j] = exp_dX * Svals[j - 1]

    # Set up lattice and terminal values
    Cvals = np.zeros((2 * N + 1, N + 1))

    # Terminal payoff for a call
    for k in range(0, 2 * N + 1):
        Cvals[k, N] = max(Svals[k] - K, 0)

    # Backward induction
    for j in range(N - 1, -1, -1):
        for i in range(N - j, N + j + 1):
            hold = discount * (p_u * Cvals[i - 1, j + 1] +
                               p_m * Cvals[i, j + 1] +
                               p_d * Cvals[i + 1, j + 1])
            # Early exercise condition for American call
            Cvals[i, j] = max(Svals[i] - K, hold)

    price = Cvals[N, 0]
    computation_time = time.time() - start_time

    return price, computation_time


In [ ]:
def RichardsonExtrapolation(P_N, P_2N, order=2):
    """
    Richardson extrapolation for error estimation
    P_N: price with N steps
    P_2N: price with 2N steps
    order: convergence order (typically 2 for trinomial trees)
    """
    return (2**order * P_2N - P_N) / (2**order - 1)

def calculate_errors(prices, benchmark_price):
    """Calculate absolute error, percentage error, and standard error"""
    abs_errors = [abs(price - benchmark_price) for price in prices]
    pct_errors = [abs_error / benchmark_price * 100 for abs_error in abs_errors]

    # Standard error of the mean (assuming prices are samples)
    if len(prices) > 1:
        std_error = np.std(prices, ddof=1) / np.sqrt(len(prices))
    else:
        std_error = 0

    return abs_errors, pct_errors, std_error

# Test parameters
S0 = 60
K = 65
sigma = 0.3
r = 0.03
T = 1.0

# Test different sample sizes
N_values = [2**6, 2**7, 2**8, 2**9, 2**10, 2**11, 2**12, 2**13]

print("=== Convergence Analysis for European Call ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

eu_call_prices = []
eu_call_times= []
for N in N_values:
    price, comp_time = TrinomialEuCall(S0, K, r, sigma, T, N)
    eu_call_prices.append(price)
    eu_call_times.append(comp_time)

# Use Richardson extrapolation as benchmark for European call
richardson_benchmark_eu_call = RichardsonExtrapolation(eu_call_prices[-2], eu_call_prices[-1])

abs_errors, pct_errors, std_error = calculate_errors(eu_call_prices, richardson_benchmark_eu_call)

for i, N in enumerate(N_values):
    print(f"{N:>8} {eu_call_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {eu_call_times[i]:>8.4f}")

print(f"\nRichardson Benchmark (European Call): {richardson_benchmark_eu_call:.6f}")
print(f"Standard Error: {std_error:.6f}")

################
print("=== Convergence Analysis for European Put ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

eu_put_prices = []
eu_put_times = []
for N in N_values:
    price, comp_time = TrinomialEuPut(S0, K, r, sigma, T, N)
    eu_put_prices.append(price)
    eu_put_times.append(comp_time)

# Use Richardson extrapolation as benchmark for European put
richardson_benchmark_eu_put = RichardsonExtrapolation(eu_put_prices[-2], eu_put_prices[-1])

abs_errors, pct_errors, std_error = calculate_errors(eu_put_prices, richardson_benchmark_eu_put)

for i, N in enumerate(N_values):
    print(f"{N:>8} {eu_put_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {eu_put_times[i]:>8.4f}")

print(f"\nRichardson Benchmark (European Put): {richardson_benchmark_eu_put:.6f}")
print(f"Standard Error: {std_error:.6f}")

################
print("\n=== Convergence Analysis for American Call ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

am_call_prices = []
am_call_times = []
for N in N_values:
    price, comp_time = TrinomialAmCall(S0, K, r, sigma, T, N)
    am_call_prices.append(price)
    am_call_times.append(comp_time)

# Use Richardson extrapolation as benchmark for American call
richardson_benchmark_am_call = RichardsonExtrapolation(am_call_prices[-2], am_call_prices[-1])

abs_errors, pct_errors, std_error = calculate_errors(am_call_prices, richardson_benchmark_am_call)

for i, N in enumerate(N_values):
    print(f"{N:>8} {am_call_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {am_call_times[i]:>8.4f}")

print(f"\nRichardson Benchmark (American Call): {richardson_benchmark_am_call:.6f}")
print(f"Standard Error: {std_error:.6f}")

################
print("\n=== Convergence Analysis for American Put ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

am_put_prices = []
am_put_times = []
for N in N_values:
    price, comp_time = TrinomialAmPut(S0, K, r, sigma, T, N)
    am_put_prices.append(price)
    am_put_times.append(comp_time)

# Use Richardson extrapolation as benchmark for American put
richardson_benchmark_am_put = RichardsonExtrapolation(am_put_prices[-2], am_put_prices[-1])

abs_errors, pct_errors, std_error = calculate_errors(am_put_prices, richardson_benchmark_am_put)

for i, N in enumerate(N_values):
    print(f"{N:>8} {am_put_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {am_put_times[i]:>8.4f}")

print(f"\nRichardson Benchmark (American Put): {richardson_benchmark_am_put:.6f}")
print(f"Standard Error: {std_error:.6f}")

=== Convergence Analysis for European Call ===
       N      Price    Abs Error Pct Error(%)  Time(s)
-----------------------------------------------------------------
      64   5.901000     0.001891       0.0321   0.0099
     128   5.901377     0.002268       0.0385   0.0394
     256   5.902716     0.003607       0.0611   0.1472
     512   5.901014     0.001905       0.0323   0.9781
    1024   5.898205     0.000904       0.0153   3.3861
    2048   5.898537     0.000572       0.0097   8.5729
    4096   5.899147     0.000038       0.0006  25.2479
    8192   5.899119     0.000010       0.0002  99.0269

Richardson Benchmark (European Call): 5.899109
Standard Error: 0.000567
=== Convergence Analysis for European Put ===
       N      Price    Abs Error Pct Error(%)  Time(s)
-----------------------------------------------------------------
      64   8.979960     0.001891       0.0211   0.0044
     128   8.980337     0.002269       0.0253   0.0174
     256   8.981676     0.003607       0.0

In [ ]:
def calculate_errors(prices, benchmark_price):
    """Calculate absolute error, percentage error, and standard error"""
    abs_errors = [abs(price - benchmark_price) for price in prices]
    pct_errors = [abs_error / benchmark_price * 100 for abs_error in abs_errors]

    if len(prices) > 1:
        std_error = np.std(prices, ddof=1) / np.sqrt(len(prices))
    else:
        std_error = 0

    return abs_errors, pct_errors, std_error

# Test parameters
S0 = 60
K = 65
sigma = 0.3
r = 0.03
T = 1.0

N_values = [2**6, 2**7, 2**8, 2**9, 2**10, 2**11, 2**12, 2**13]

################
print("=== Convergence Analysis for European Call ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

eu_call_prices = []
eu_call_times = []
for N in N_values:
    price, comp_time = TrinomialEuCall(S0, K, r, sigma, T, N)
    eu_call_prices.append(price)
    eu_call_times.append(comp_time)

if len(eu_call_prices) >= 3:
    P_N = eu_call_prices[-3]
    P_2N = eu_call_prices[-2]
    P_4N = eu_call_prices[-1]

    R1_call = RichardsonExtrapolation(P_N, P_2N)
    R2_call = RichardsonExtrapolation(P_2N, P_4N)
    higher_order_estimate_eu_call = (4 * R2_call - R1_call) / 3

    print("\nEuropean Call Higher-Order Richardson:")
    print(f"Second-order Richardson (2^11, 2^12): {R1_call:.6f}")
    print(f"Second-order Richardson (2^12, 2^13): {R2_call:.6f}")
    print(f"Third-order Richardson estimate: {higher_order_estimate_eu_call:.6f}")

    abs_errors, pct_errors, std_error = calculate_errors(eu_call_prices, higher_order_estimate_eu_call)

    for i, N in enumerate(N_values):
        print(f"{N:>8} {eu_call_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {eu_call_times[i]:>8.4f}")

    print(f"\nRichardson Benchmark (European Call): {higher_order_estimate_eu_call:.6f}")
    print(f"Standard Error: {std_error:.6f}")

################
print("=== Convergence Analysis for European Put ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

eu_put_prices = []
eu_put_times = []
for N in N_values:
    price, comp_time = TrinomialEuPut(S0, K, r, sigma, T, N)
    eu_put_prices.append(price)
    eu_put_times.append(comp_time)

if len(eu_put_prices) >= 3:
    P_N = eu_put_prices[-3]
    P_2N = eu_put_prices[-2]
    P_4N = eu_put_prices[-1]

    R1_put = RichardsonExtrapolation(P_N, P_2N)
    R2_put = RichardsonExtrapolation(P_2N, P_4N)
    higher_order_estimate_eu_put = (4 * R2_put - R1_put) / 3

    print("\nEuropean Put Higher-Order Richardson:")
    print(f"Second-order Richardson (2^11, 2^12): {R1_put:.6f}")
    print(f"Second-order Richardson (2^12, 2^13): {R2_put:.6f}")
    print(f"Third-order Richardson estimate: {higher_order_estimate_eu_put:.6f}")

    abs_errors, pct_errors, std_error = calculate_errors(eu_put_prices, higher_order_estimate_eu_put)

    for i, N in enumerate(N_values):
        print(f"{N:>8} {eu_put_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {eu_put_times[i]:>8.4f}")

    print(f"\nRichardson Benchmark (European Put): {higher_order_estimate_eu_put:.6f}")
    print(f"Standard Error: {std_error:.6f}")

################
print("\n=== Convergence Analysis for American Call ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

am_call_prices = []
am_call_times = []
for N in N_values:
    price, comp_time = TrinomialAmCall(S0, K, r, sigma, T, N)
    am_call_prices.append(price)
    am_call_times.append(comp_time)

if len(am_call_prices) >= 3:
    P_N = am_call_prices[-3]
    P_2N = am_call_prices[-2]
    P_4N = am_call_prices[-1]

    R1_am_call = RichardsonExtrapolation(P_N, P_2N)
    R2_am_call = RichardsonExtrapolation(P_2N, P_4N)
    higher_order_estimate_am_call = (4 * R2_am_call - R1_am_call) / 3

    print("\nAmerican Call Higher-Order Richardson:")
    print(f"Second-order Richardson (2^11, 2^12): {R1_am_call:.6f}")
    print(f"Second-order Richardson (2^12, 2^13): {R2_am_call:.6f}")
    print(f"Third-order Richardson estimate: {higher_order_estimate_am_call:.6f}")

    abs_errors, pct_errors, std_error = calculate_errors(am_call_prices, higher_order_estimate_am_call)

    for i, N in enumerate(N_values):
        print(f"{N:>8} {am_call_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {am_call_times[i]:>8.4f}")

    print(f"\nRichardson Benchmark (American Call): {higher_order_estimate_am_call:.6f}")
    print(f"Standard Error: {std_error:.6f}")

################
print("\n=== Convergence Analysis for American Put ===")
print(f"{'N':>8} {'Price':>10} {'Abs Error':>12} {'Pct Error(%)':>12} {'Time(s)':>8}")
print("-" * 65)

am_put_prices = []
am_put_times = []
for N in N_values:
    price, comp_time = TrinomialAmPut(S0, K, r, sigma, T, N)
    am_put_prices.append(price)
    am_put_times.append(comp_time)

if len(am_put_prices) >= 3:
    P_N = am_put_prices[-3]
    P_2N = am_put_prices[-2]
    P_4N = am_put_prices[-1]

    R1_am_put = RichardsonExtrapolation(P_N, P_2N)
    R2_am_put = RichardsonExtrapolation(P_2N, P_4N)
    higher_order_estimate_am_put = (4 * R2_am_put - R1_am_put) / 3

    print("\nAmerican Put Higher-Order Richardson:")
    print(f"Second-order Richardson (2^11, 2^12): {R1_am_put:.6f}")
    print(f"Second-order Richardson (2^12, 2^13): {R2_am_put:.6f}")
    print(f"Third-order Richardson estimate: {higher_order_estimate_am_put:.6f}")

    abs_errors, pct_errors, std_error = calculate_errors(am_put_prices, higher_order_estimate_am_put)

    for i, N in enumerate(N_values):
        print(f"{N:>8} {am_put_prices[i]:>10.6f} {abs_errors[i]:>12.6f} {pct_errors[i]:>12.4f} {am_put_times[i]:>8.4f}")

    print(f"\nRichardson Benchmark (American Put): {higher_order_estimate_am_put:.6f}")
    print(f"Standard Error: {std_error:.6f}")

=== Convergence Analysis for European Call ===
       N      Price    Abs Error Pct Error(%)  Time(s)
-----------------------------------------------------------------

European Call Higher-Order Richardson:
Second-order Richardson (2^11, 2^12): 5.899351
Second-order Richardson (2^12, 2^13): 5.899109
Third-order Richardson estimate: 5.899028
      64   5.901000     0.001972       0.0334   0.0043
     128   5.901377     0.002349       0.0398   0.0169
     256   5.902716     0.003688       0.0625   0.0752
     512   5.901014     0.001986       0.0337   0.3056
    1024   5.898205     0.000823       0.0140   1.2042
    2048   5.898537     0.000491       0.0083   5.2800
    4096   5.899147     0.000119       0.0020  23.8255
    8192   5.899119     0.000090       0.0015  97.9544

Richardson Benchmark (European Call): 5.899028
Standard Error: 0.000567
=== Convergence Analysis for European Put ===
       N      Price    Abs Error Pct Error(%)  Time(s)
------------------------------------------